In [ ]:
import pandas as pd

df = pd.read_csv(r"C:\Users\Pratik Kumar\Desktop\@mai\ObesityDataSet_raw_and_data_sinthetic.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df.dtypes

In [ ]:
df.columns

In [ ]:
df['NObeyesdad'].value_counts()

In [ ]:
categorical_features = df.select_dtypes(include='str').columns.tolist()
categorical_features

In [ ]:
for feature in categorical_features:
    print(f'{feature} : {df[feature].nunique()}')

In [ ]:
for feature in categorical_features:
    percentage = (((df[feature].value_counts()) / len(df)) * 100).round(2)
    print(percentage.astype(str) + '%')
    print()

In [ ]:
numerical_features = df.select_dtypes(include='number').columns
numerical_features

In [ ]:
df[numerical_features].describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

n_rows = 2
n_cols = 5

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
axes = axes.flatten()

for index, feature in enumerate(numerical_features):
    sns.boxplot(data=df[feature], ax=axes[index])
    axes[index].set_title(feature)
    axes[index].set_ylabel('Values')
    axes[index].grid(True)

for i in range(len(numerical_features), len(axes)):
    plt.delaxes(axes[i])

plt.tight_layout()
plt.show()

In [ ]:
results = []

for feature in numerical_features:
    data = df[feature]
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = ((data < lower_bound) | (data > upper_bound)).sum()
    outliers_percentage = ((outliers / len(data)) * 100).round(2)
    results.append({'Numerical_Features' : feature, 'Outlier_Count' : outliers, 'Outliers_Percentage' : outliers_percentage})

pd.DataFrame(results)

In [ ]:
n_rows = 2
n_cols = 5

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
axes = axes.flatten()

for index, feature in enumerate(numerical_features):
    sns.kdeplot(data=df[feature], ax=axes[index], fill=True)
    axes[index].set_ylabel('Density')
    axes[index].grid(True)

for i in range(len(numerical_features), len(axes)):
    plt.delaxes(axes[i])

plt.tight_layout()
plt.show()

In [ ]:
skewness = []

for feature in numerical_features:
    skewness.append({'Numerical_Features' : feature, 'Skewness_Value' : df[feature].skew()})

pd.DataFrame(skewness)

In [ ]:
from sklearn.model_selection import train_test_split

X = df.iloc[:, :16]
Y = df.iloc[:, 16]
X_train, X_temp, Y_train, Y_temp = train_test_split(X, Y, test_size=0.3, random_state=42, stratify=Y)
X_test, X_val, Y_test, Y_val = train_test_split(X_temp, Y_temp, test_size=0.5, random_state=42, stratify=Y_temp)

print("Training Data Shape :", X_train.shape, Y_train.shape)
print("Validation Data Shape :", X_val.shape, Y_val.shape)
print("Testing Data Shape :", X_test.shape, Y_test.shape)

In [ ]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

oe_columns = ["Gender", "family_history_with_overweight", "FAVC", "SMOKE", "SCC"]
ohe_columns = ["CAEC", "CALC", "MTRANS"]

oe = OrdinalEncoder()
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

X_train[oe_columns] = oe.fit_transform(X_train[oe_columns])
X_val[oe_columns] = oe.transform(X_val[oe_columns])
X_test[oe_columns] = oe.transform(X_test[oe_columns])

X_train_ohe = pd.DataFrame(
    ohe.fit_transform(X_train[ohe_columns]),
    columns=ohe.get_feature_names_out(ohe_columns),
    index=X_train.index
)
X_val_ohe = pd.DataFrame(
    ohe.transform(X_val[ohe_columns]),
    columns=ohe.get_feature_names_out(ohe_columns),
    index=X_val.index
)
X_test_ohe = pd.DataFrame(
    ohe.transform(X_test[ohe_columns]),
    columns=ohe.get_feature_names_out(ohe_columns),
    index=X_test.index
)

X_train = pd.concat([X_train.drop(columns=ohe_columns), X_train_ohe], axis=1)
X_val = pd.concat([X_val.drop(columns=ohe_columns), X_val_ohe], axis=1)
X_test = pd.concat([X_test.drop(columns=ohe_columns), X_test_ohe], axis=1)

print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Testing shape:", X_test.shape)

In [ ]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

models = {
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "Extra Trees": ExtraTreesClassifier(n_estimators=200, random_state=42, n_jobs=-1)
}

for model_name, model in models.items():
    model.fit(X_train, Y_train)
    print(f"{model_name} training completed.")

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

datasets = {
    "Train": (X_train, Y_train),
    "Validation": (X_val, Y_val),
    "Test": (X_test, Y_test)
}

performance_results = []

for model_name, model in models.items():
    for split_name, (X_split, Y_split) in datasets.items():
        Y_pred = model.predict(X_split)

        performance_results.append({
            "Model": model_name,
            "Dataset": split_name,
            "Accuracy": accuracy_score(Y_split, Y_pred),
            "Macro-F1": f1_score(Y_split, Y_pred, average="macro")
        })

performance_df = pd.DataFrame(performance_results)

display(
    performance_df.style
    .format({
        "Accuracy": "{:.4f}",
        "Macro-F1": "{:.4f}"
    })
    .set_properties(**{"text-align": "center"})
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "center")]}
    ])
)

## Confusion Matrix and Class-wise Evaluation

Evaluate both trained models on the same held-out test set. The confusion matrices show the number of correct and incorrect predictions for each obesity-level class. The classification reports provide precision, recall, and F1-score for every class.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# Keep the same class order for both models
class_labels = sorted(Y_test.unique())

# Generate test predictions for both classifiers
rf_test_predictions = models["Random Forest"].predict(X_test)
et_test_predictions = models["Extra Trees"].predict(X_test)

# Calculate confusion matrices
rf_cm = confusion_matrix(
    Y_test, rf_test_predictions, labels=class_labels
)
et_cm = confusion_matrix(
    Y_test, et_test_predictions, labels=class_labels
)

# Display both confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

sns.heatmap(
    rf_cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_labels,
    yticklabels=class_labels,
    ax=axes[0]
)
axes[0].set_title("Random Forest - Confusion Matrix")
axes[0].set_xlabel("Predicted Class")
axes[0].set_ylabel("Actual Class")
axes[0].tick_params(axis="x", rotation=45)
axes[0].tick_params(axis="y", rotation=0)

sns.heatmap(
    et_cm,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=class_labels,
    yticklabels=class_labels,
    ax=axes[1]
)
axes[1].set_title("Extra Trees - Confusion Matrix")
axes[1].set_xlabel("Predicted Class")
axes[1].set_ylabel("Actual Class")
axes[1].tick_params(axis="x", rotation=45)
axes[1].tick_params(axis="y", rotation=0)

plt.tight_layout()
plt.show()

# Print class-wise precision, recall and F1-score for each model
print("RANDOM FOREST - CLASSIFICATION REPORT")
print(
    classification_report(
        Y_test,
        rf_test_predictions,
        labels=class_labels,
        target_names=class_labels,
        zero_division=0
    )
)

print("EXTRA TREES - CLASSIFICATION REPORT")
print(
    classification_report(
        Y_test,
        et_test_predictions,
        labels=class_labels,
        target_names=class_labels,
        zero_division=0
    )
)


In [ ]:
X_train.columns

### SHAP dependency check
Run this cell before the SHAP explainability section. It checks whether SHAP is available in the **current Jupyter kernel** and installs it if it is missing.

In [ ]:
import sys
import subprocess
import importlib.util

if importlib.util.find_spec('shap') is None:
    print('SHAP is not installed in this notebook environment. Installing it now...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'shap'])
    print('SHAP installation completed. Now run the next SHAP import cell.')
else:
    print('SHAP is already installed.')

In [ ]:
import numpy as np
import shap
from scipy.sparse import issparse

feature_names = X_train.columns.tolist()

X_test_array = X_test.toarray() if issparse(X_test) else np.asarray(X_test)
X_test = pd.DataFrame(X_test_array, columns=feature_names)

def calculate_shap_importance(model, X_data):
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_data)

    # Handle multiclass SHAP output
    if isinstance(shap_values, list):
        shap_values = np.stack(shap_values, axis=-1)
    else:
        shap_values = np.asarray(shap_values)

    # Average absolute SHAP values across samples and classes
    if shap_values.ndim == 3:
        importance = np.abs(shap_values).mean(axis=(0, 2))
    else:
        importance = np.abs(shap_values).mean(axis=0)

    return pd.Series(
        importance,
        index=X_data.columns
    ).sort_values(ascending=False)

rf_shap = calculate_shap_importance(
    models["Random Forest"], X_test
)
et_shap = calculate_shap_importance(
    models["Extra Trees"], X_test
)

pd.DataFrame({
    "Random Forest": rf_shap,
    "Extra Trees": et_shap
}).fillna(0).round(4)

In [ ]:
def group_feature_importance(importance):
    grouped = {}

    for feature, value in importance.items():
        if feature.startswith("CAEC_"):
            original_feature = "CAEC"
        elif feature.startswith("CALC_"):
            original_feature = "CALC"
        elif feature.startswith("MTRANS_"):
            original_feature = "MTRANS"
        else:
            original_feature = feature

        grouped[original_feature] = (
            grouped.get(original_feature, 0) + value
        )

    return pd.Series(grouped).sort_values(ascending=False)

rf_grouped = group_feature_importance(rf_shap)
et_grouped = group_feature_importance(et_shap)

pd.DataFrame({
    "Random Forest": rf_grouped,
    "Extra Trees": et_grouped
}).fillna(0).round(4)

In [ ]:
rf_top5 = set(rf_grouped.nlargest(5).index)
et_top5 = set(et_grouped.nlargest(5).index)

common_features = rf_top5 & et_top5
top5_overlap = len(common_features) / 5 * 100

print("Random Forest Top 5:", list(rf_grouped.nlargest(5).index))
print("Extra Trees Top 5:", list(et_grouped.nlargest(5).index))
print("Common Top 5:", sorted(common_features))
print(f"Top-5 Feature Overlap: {top5_overlap:.2f}%")

In [ ]:
from scipy.stats import spearmanr

all_features = sorted(set(rf_grouped.index) | set(et_grouped.index))

rf_values = rf_grouped.reindex(all_features).fillna(0)
et_values = et_grouped.reindex(all_features).fillna(0)

correlation, p_value = spearmanr(rf_values, et_values)

print(f"Spearman Rank Correlation: {correlation:.4f}")
print(f"P-value: {p_value:.6f}")

In [ ]:
# Use the exact feature order used to train the models
feature_names = X_train.columns.tolist()
X_test_array = X_test.toarray() if issparse(X_test) else np.asarray(X_test)
X_test = pd.DataFrame(X_test_array, columns=feature_names)

# Map encoded columns to their original feature names
def original_feature_name(name):
    for prefix in ["CAEC_", "CALC_", "MTRANS_"]:
        if name.startswith(prefix):
            return prefix[:-1]
    return name

feature_groups = {}
for i, name in enumerate(feature_names):
    original = original_feature_name(name)
    feature_groups.setdefault(original, []).append(i)

def calculate_grouped_shap(model, X_data):
    explainer = shap.TreeExplainer(model)
    values = explainer.shap_values(X_data)

    if isinstance(values, list):
        # Older SHAP versions: one array per class
        values = np.stack(values, axis=-1)
    else:
        values = np.asarray(values)

    if values.ndim == 2:
        values = values[:, :, np.newaxis]

    # Sum encoded-column contributions first, then take absolute values
    grouped_importance = {}
    for feature, indices in feature_groups.items():
        grouped_values = values[:, indices, :].sum(axis=1)
        grouped_importance[feature] = np.abs(grouped_values).mean()

    return pd.Series(grouped_importance).sort_values(ascending=False)

rf_grouped = calculate_grouped_shap(
    models["Random Forest"], X_test
)
et_grouped = calculate_grouped_shap(
    models["Extra Trees"], X_test
)

grouped_comparison = pd.DataFrame({
    "Random Forest": rf_grouped,
    "Extra Trees": et_grouped
}).fillna(0)

display(grouped_comparison.sort_values(
    "Random Forest", ascending=False
).round(4))

In [ ]:
plot_df = grouped_comparison.sort_values(
    "Random Forest", ascending=True
)

ax = plot_df.plot(
    kind="barh",
    figsize=(10, 8),
    width=0.8
)

ax.set_title("Grouped SHAP Feature Importance Comparison")
ax.set_xlabel("Mean Absolute Grouped SHAP Value")
ax.set_ylabel("Original Feature")
ax.legend(title="Model")
plt.tight_layout()
plt.show()

In [ ]:
rf_top5 = set(rf_grouped.nlargest(5).index)
et_top5 = set(et_grouped.nlargest(5).index)
common_features = rf_top5 & et_top5

top5_overlap = len(common_features) / 5 * 100

all_features = sorted(
    set(rf_grouped.index) | set(et_grouped.index)
)
rf_ranks = rf_grouped.reindex(all_features).fillna(0).rank(
    ascending=False, method="average"
)
et_ranks = et_grouped.reindex(all_features).fillna(0).rank(
    ascending=False, method="average"
)

correlation, p_value = spearmanr(rf_ranks, et_ranks)

print("Random Forest Top 5:", list(rf_grouped.nlargest(5).index))
print("Extra Trees Top 5:", list(et_grouped.nlargest(5).index))
print("Common Top 5:", sorted(common_features))
print(f"Top-5 Feature Overlap: {top5_overlap:.2f}%")
print(f"Spearman Rank Correlation: {correlation:.4f}")
print(f"P-value: {p_value:.6f}")

In [ ]:
final_results = pd.DataFrame([
    {
        "Model": "Random Forest",
        "Test Accuracy": 0.9521,
        "Test Macro-F1": 0.9515,
        "Top-5 Overlap (%)": top5_overlap,
        "Spearman Correlation": correlation
    },
    {
        "Model": "Extra Trees",
        "Test Accuracy": 0.8978,
        "Test Macro-F1": 0.8972,
        "Top-5 Overlap (%)": top5_overlap,
        "Spearman Correlation": correlation
    }
])

final_results.round(4)